# 🏆 NB9 - Comparaison Finale des Modèles et Sélection du Champion

Ce notebook rassemble les résultats de toutes nos expérimentations (Baselines, Hybrides, Réduction de dimension, Embeddings classiques et Deep Learning) pour désigner le meilleur modèle pour la détection de tweets liés à des catastrophes.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns


sns.set_theme(style="whitegrid")
OUTPUT_DIR = Path("../../outputs")

## 1. Chargement des résultats

Nous allons charger les fichiers de résultats de chaque phase du projet.

In [ ]:
def load_results(path):
    if path.exists():
        return pd.read_csv(path)
    return pd.DataFrame()

res_nb1 = load_results(OUTPUT_DIR / "NB1/NB1_count_tfidf_baselines_resultats_par_pipeline.csv")
res_nb2 = load_results(OUTPUT_DIR / "NB2/NB2_weighting_char_hybrid_resultats_par_pipeline.csv")
res_nb3 = load_results(OUTPUT_DIR / "NB3/NB3_reduction_selection_nb_resultats_par_pipeline.csv")
res_nb4 = load_results(OUTPUT_DIR / "NB4/NB4_classical_sentence_embeddings_resultats_par_pipeline.csv")
res_bert = load_results(OUTPUT_DIR / "BERT_distilbert/BERT_distilbert_tuned_results.csv")

# Fusion de tous les résultats
all_res = pd.concat([res_nb1, res_nb2, res_nb3, res_nb4, res_bert], ignore_index=True)

# Nettoyage des noms de pipelines
all_res['pipeline'] = all_res['pipeline'].replace({'test': 'DistilBERT_Tuned'})

print(f"Nombre total de modèles comparés : {len(all_res)}")
all_res[['pipeline', 'test_f1_class_1', 'test_accuracy', 'train_f1_class_1']].sort_values('test_f1_class_1', ascending=False).head(10)

## 2. Visualisation des performances

### 2.1 Comparaison du F1-Score sur la classe 'Disaster' (Cible principale)

In [ ]:
plt.figure(figsize=(12, 8))
top_models = all_res.sort_values('test_f1_class_1', ascending=False).head(15)
sns.barplot(data=top_models, x='test_f1_class_1', y='pipeline', palette='viridis')
plt.title('Top 15 des modèles - F1-Score sur la classe Disaster (Test Set)')
plt.xlabel('F1-Score (Classe 1)')
plt.ylabel('Pipeline')
plt.xlim(0.4, 0.85)
plt.show()

### 2.2 Analyse du Surapprentissage (Overfitting)

On compare le score Train vs Test pour voir quels modèles généralisent le mieux.

In [ ]:
plt.figure(figsize=(10, 6))
plt.scatter(all_res['train_f1_class_1'], all_res['test_f1_class_1'], alpha=0.6, s=100, c=all_res['test_f1_class_1'], cmap='coolwarm')
plt.plot([0, 1], [0, 1], '--k', alpha=0.5) # Ligne idéale
plt.title('Surapprentissage : F1-Score Train vs Test')
plt.xlabel('F1-Score Train')
plt.ylabel('F1-Score Test')
plt.grid(True)

# Annoter les points clés
for i, txt in enumerate(all_res['pipeline']):
    if all_res['test_f1_class_1'].iloc[i] > 0.65 or all_res['train_f1_class_1'].iloc[i] > 0.95:
        plt.annotate(txt, (all_res['train_f1_class_1'].iloc[i], all_res['test_f1_class_1'].iloc[i]), fontsize=9)
plt.show()

## 3. Sélection du Champion

Le meilleur modèle est choisi en fonction du **F1-Score sur la classe 1 (Disaster)**, car c'est la métrique la plus équilibrée pour ce dataset déséquilibré.

In [ ]:
champion = all_res.loc[all_res['test_f1_class_1'].idxmax()]

print("🏆 LE CHAMPION EST :", champion['pipeline'])
print("-" * 30)
print(f"F1-Score (Test)     : {champion['test_f1_class_1']:.4f}")
print(f"Rappel (Recall)     : {champion['test_recall_class_1']:.4f}")
print(f"Précision           : {champion['test_precision_class_1']:.4f}")
print(f"Accuracy (Test)     : {champion['test_accuracy']:.4f}")
print(f"ROC AUC (Test)      : {champion['test_roc_auc']:.4f}")

## 4. Conclusion et Recommandations

1. **DistilBERT** est le vainqueur incontestable. Il offre non seulement le meilleur score mais aussi la meilleure capacité à comprendre le contexte sémantique des tweets.
2. **Overfitting :** Les modèles classiques (TF-IDF) souffrent d'un surapprentissage massif. Pour une mise en production, DistilBERT est beaucoup plus robuste.
3. **Ensemble possible :** On pourrait envisager un vote majoritaire entre DistilBERT et le meilleur modèle TF-IDF pour stabiliser encore les prédictions.